# Going further — feature neutralization & target ensembling (EIQ)

Two techniques to push past the `hello_eiq` baseline, both aimed at **AIMC** (the 3x-weighted payout term):

- **Feature neutralization** (Part A) — strip a model's exposure to risky feature groups.
- **Target ensembling** (Part B) — train on several diverse targets and ensemble the results.

Assumes you've worked through [`hello_eiq.ipynb`](hello_eiq.ipynb). We redo the minimal setup, train a base model, then explore each technique.

> AIMC (alpha over the live stake-weighted consensus) is finalized **server-side**. Offline we measure **CORR**, **feature exposure**, and a **contribution-over-benchmarks** proxy (BMC-style) built from the downloadable benchmark predictions; the true AIMC is confirmed after submitting via `client.run_validation_diagnostics(...)`.
>
> This notebook trains several models and runs per-exped loops — expect a few minutes end to end.

## Setup

This section rebuilds the minimal pipeline from `hello_eiq` so the notebook stands on its own, then trains the one model both techniques start from. In order, we:

1. **install & authenticate** against the closed-beta API,
2. pull the **schema** and **download** train + validation,
3. **embargo** the validation split (the same 20-day leakage guard as `hello_eiq`),
4. define a small **evaluation toolkit** — per-exped CORR, `neutralize()`, and a `contribution()` proxy for AIMC, and
5. train a single **base model** on `target_everest_20`.

The only genuinely new pieces (vs `hello_eiq`) are the `neutralize()` and `contribution()` helpers, which power both Part A and Part B.

### 1. Install & authenticate

Same dependencies and the same env-var login as `hello_eiq` — credentials are read from the environment, never written into the notebook.

In [ ]:
# everestapi is the EIQ SDK; the rest are for training, evaluation, and plotting.
%pip install --quiet "everestapi>=0.2.1" lightgbm pandas pyarrow scikit-learn scipy matplotlib

In [ ]:
import os
from everestapi import EverestAPI

# Read every credential from the environment (set via onboarding's "Copy setup command").
base_url = os.environ.get("EIQ_BASE_URL", "https://staging.everesteer.ai")
api_key = os.environ.get("EIQ_API_KEY") or os.environ.get("EVEREST_API_KEY")
cf_id = os.environ.get("CF_ACCESS_CLIENT_ID")
cf_secret = os.environ.get("CF_ACCESS_CLIENT_SECRET")

# Fail fast with a clear message instead of a cryptic 401/403 later on.
if not api_key:
    raise RuntimeError("Set EIQ_API_KEY (onboarding -> Copy setup command).")
if "staging" in base_url and not (cf_id and cf_secret):
    raise RuntimeError("Closed beta: set CF_ACCESS_CLIENT_ID / CF_ACCESS_CLIENT_SECRET.")

client = EverestAPI(api_key=api_key, base_url=base_url,
                    cf_access_client_id=cf_id, cf_access_client_secret=cf_secret)
client.health()  # {'status': 'ok', ...} on a working, authenticated connection

### 2. Load the data

We pull the schema (feature sets, groups, targets), download `train`/`validation`, and build `val_eval` — the rows every model is scored on. Two deliberate choices:

- **Embargo** — drop validation expeds within the target's 20-day window of the last training exped, so the 20-day forward returns of train and validation never overlap (the data-leakage guard from `hello_eiq` Section 7).
- **Downsampling** — keep every 4th exped of *both* train and validation, purely so this multi-model notebook runs in minutes; comment those lines out for a full, more accurate run.

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# The schema is the map of the data: feature sets/groups and the list of targets.
schema = client.get_dataset_schema()
feature_sets = schema["feature_sets"]
target_cols = schema["targets"]
PRIMARY_TARGET = "target_everest_20"           # the scored / payout target
EXPED = "exped"                                # one exped = one trading day
feature_set = feature_sets["small"]            # 329 features; keep it light (we train many models)

# Download the two splits we need (live is only needed at submission time).
train = pd.read_parquet(client.download_dataset("futures", "train"))
val = pd.read_parquet(client.download_dataset("futures", "validation"))

def exped_num(e):
    # Expeds are strings like "exped_8978"; pull the integer so we can order them in time.
    return int(str(e).split("_")[-1])

# Speed: keep every 4th TRAIN exped. (Comment out to use the full dataset.)
train = train[train[EXPED].isin(sorted(train[EXPED].unique(), key=exped_num)[::4])]

def features_matrix(df, cols):
    # Cast to float and turn the -1 missing sentinel into NaN (LightGBM handles NaN natively).
    X = df[cols].astype("float32")
    return X.where(X >= 0)

# Build the evaluation set:
#  - keep only rows that have targets (data_type == "validation"),
#  - EMBARGO: drop expeds within 20 of the last train exped (20-day forward-return overlap),
#  - reset_index so `id` is a column AND the index is a fresh unique RangeIndex. That last
#    point matters: instrument ids repeat across expeds, and the per-exped routines below
#    stitch their results back together by index, so a non-unique index would misalign them.
EMBARGO = 20
last_train = max(train[EXPED].map(exped_num))
val_eval = val[(val["data_type"] == "validation") &
               (val[EXPED].map(exped_num) > last_train + EMBARGO)].reset_index()
# Speed: keep every 4th VALIDATION exped too. (Comment out to use the full dataset.)
val_eval = val_eval[val_eval[EXPED].isin(
    sorted(val_eval[EXPED].unique(), key=exped_num)[::4])].reset_index(drop=True)

def per_exped_corr(df, pred_col, target_col=PRIMARY_TARGET):
    # Rank (Spearman) correlation of predictions with the target, one value per exped.
    # Sorted chronologically by exped so cumulative-sum plots read like a backtest.
    out = {}
    for e, g in df.groupby(EXPED):
        rho, _ = spearmanr(g[pred_col], g[target_col])
        if np.isfinite(rho):
            out[e] = rho
    return pd.Series(out).sort_index(key=lambda idx: idx.map(exped_num))

def metrics(corr):
    # Numerai-style summary of a per-exped CORR series: mean, volatility, Sharpe, drawdown.
    # Used everywhere we report prediction performance, so the columns are consistent.
    cum = corr.cumsum()
    return {"mean_CORR": corr.mean(),
            "std": corr.std(),
            "sharpe": corr.mean() / corr.std(),
            "max_drawdown": (cum.expanding().max() - cum).max()}

print("setup:", train.shape, "train |", val_eval.shape, "val |", val_eval[EXPED].nunique(), "expeds")

### 3. Evaluation toolkit

Two helpers do the heavy lifting throughout this notebook:

- **`neutralize(preds, neutralizers, proportion)`** — removes the part of your predictions that a set of columns can linearly explain (a least-squares projection), leaving the residual. Part A uses it to strip feature exposure; it also powers the contribution proxy below.
- **`contribution(df, pred_col)`** — an **offline, BMC-style stand-in for AIMC**. AIMC rewards being *different from the consensus yet still right*, so we residualize predictions against the **benchmark consensus** (the mean of the downloadable benchmark models, which fully covers validation) and correlate that residual with the target. It points the same direction as AIMC; the real number comes from the server after you submit.

In [ ]:
# neutralize(): the projection used by both feature neutralization and the AIMC proxy.
def neutralize(preds, neutralizers, proportion=1.0):
    # Remove `proportion` of the part of `preds` that `neutralizers` can linearly explain,
    # via a least-squares projection; return the residual (lower-exposure) predictions.
    y = preds.to_numpy(dtype="float64").reshape(-1, 1)
    X = neutralizers.to_numpy(dtype="float64")
    X = np.hstack([X, np.ones((len(X), 1))])              # add an intercept column
    projection = X @ (np.linalg.pinv(X, rcond=1e-6) @ y)  # least-squares fit, then project
    return pd.Series((y - proportion * projection).ravel(), index=preds.index)

# The "consensus" we measure contribution against: the mean of the benchmark models'
# predictions over validation (verified to cover every validation id + exped).
bench = pd.read_parquet(client.download_benchmark("futures", "validation")).reset_index()
bench_cols = [c for c in bench.columns if c not in ("id", EXPED)]
bench["consensus"] = bench[bench_cols].mean(axis=1)
val_eval = val_eval.merge(bench[["id", EXPED, "consensus"]], on=["id", EXPED], how="left")

def contribution(df, pred_col, target_col=PRIMARY_TARGET):
    # Offline AIMC proxy: residualize predictions against the consensus (per exped),
    # then correlate the residual with the target. High contribution = adds signal the
    # consensus doesn't already have. True AIMC (vs the live consensus) is confirmed post-submit.
    out = {}
    for e, g in df.groupby(EXPED):
        resid = neutralize(g[pred_col], g[["consensus"]])
        rho, _ = spearmanr(resid, g[target_col])
        if np.isfinite(rho):
            out[e] = rho
    return pd.Series(out).sort_index(key=lambda idx: idx.map(exped_num))

print(f"benchmark consensus matched {val_eval['consensus'].notna().mean() * 100:.0f}% of rows "
      f"({len(bench_cols)} benchmark models)")

### 4. Base model

A single LightGBM trained on `target_everest_20` — the starting point for both techniques. We print its CORR and contribution (AIMC proxy) so every later result has a baseline to beat.

In [ ]:
# A modest, lightly-regularized gradient-boosted tree (same recipe as hello_eiq).
def make_model():
    return lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.01, max_depth=6,
                             num_leaves=64, colsample_bytree=0.1, min_child_samples=500,
                             random_state=42, verbose=-1)

base = make_model().fit(features_matrix(train, feature_set), train[PRIMARY_TARGET])
val_eval["pred_base"] = base.predict(features_matrix(val_eval, feature_set))

# Baseline metrics that Parts A and B are compared against (same columns used throughout).
base_perf = pd.DataFrame({"base (everest_20)": {
    **metrics(per_exped_corr(val_eval, "pred_base")),
    "contribution": contribution(val_eval, "pred_base").mean()}}).T
display(base_perf)

## Part A — Feature neutralization

A model often leans heavily on a few features; if those stop working, its edge can vanish — that's **feature risk**. **Feature exposure** is the correlation between your predictions and each feature. **Neutralization** removes the part of your predictions explained by a chosen set of features, leaving a lower-exposure residual — often trading a little CORR for lower feature risk and, frequently, steadier performance or better contribution (AIMC).

We use the `neutralize()` helper from setup (a least-squares projection).

### 1. Feature groups

You can neutralize against **any** set of features — a single feature, an arbitrary subset, or a whole named group (you can even neutralize against another signal entirely, which is exactly what `contribution()` does against the consensus). Groups are just a convenient, interpretable way to *choose* a set, so they're the menu we'll demo with.

**What's a group?** EIQ organizes its features into named groups (`cirque`, `serac`, `glacier`, ...). A group collects features that measure a *similar kind of thing* — think of it as a theme or family of related signals. The exact meaning is obfuscated, but the grouping is meaningful: neutralizing against a whole group removes the model's collective lean on that entire theme, rather than on one isolated feature.

In the table below, `group_size` is the total number of features in the group; `in_small` is how many of them the **base model actually trained on** (it used the `small` set) — a group can include features the model never saw, which you can still neutralize against.

In [ ]:
# The named groups are every feature_sets key that isn't a size tier (small/medium/all).
group_names = [g for g in feature_sets if g not in ("small", "medium", "all")]
# For each group: its total feature count, and how many of those are in the 'small' set
# the base model was trained on.
pd.DataFrame({
    "group_size": {g: len(feature_sets[g]) for g in group_names},
    "in_small":   {g: len(set(feature_sets[g]) & set(feature_sets["small"])) for g in group_names},
}).sort_values("group_size", ascending=False)

### 2. Feature exposure

First, *see* the problem: how strongly does the base model lean on each feature in a group?

**How we measure it.** Feature exposure is simply the **correlation between the model's predictions and a feature**, computed across the validation rows. For every feature in the chosen group we correlate it with `pred_base` — one number per feature:

- a large **positive** value -> the model's output rises with that feature,
- a large **negative** value -> it moves opposite,
- near **zero** -> the model barely uses it.

We care about the **magnitude** (`|exposure|`), so we report the **max** (the single feature the model depends on most — its biggest concentrated risk) and the **mean** (its overall lean on the group). Sorting the exposures and plotting them reveals the *shape* of that risk: a few tall bars at the ends mean the model's bets are concentrated in a handful of features (fragile if those stop working), whereas a flat profile means the exposure is spread out.

In [ ]:
# Pick one group to demonstrate on (any group works — see the menu above).
GROUP = "serac"
# The group's feature columns that are actually present in our data.
group_feats = [f for f in feature_sets[GROUP] if f in val_eval.columns]

# Feature exposure = how correlated the model's predictions are with each feature.
# A large |value| means the model leans heavily on that feature -> concentrated risk.
exposure = val_eval[group_feats].corrwith(val_eval["pred_base"])

# Plot the exposures sorted low->high; the tall bars at the two ends are the risk concentration.
exposure.sort_values().plot(
    kind="bar", figsize=(10, 3), xticks=[], xlabel="features (sorted)",
    title=f"Base-model exposure to group '{GROUP}' ({len(group_feats)} features)"
)
plt.tight_layout(); plt.show()

# Two summary numbers: the single worst exposure, and the average across the group.
print(f"max |exposure|: {exposure.abs().max():.3f} | mean |exposure|: {exposure.abs().mean():.3f}")

### 3. Neutralization proportion

`proportion` is a dial from 0 to 1 that controls **how much** of the feature-explained signal to strip out:

- `proportion = 0` — do nothing; predictions are unchanged.
- `proportion = 1` — remove *all* of the signal those features can linearly explain, driving the model's exposure to them toward 0.
- in between — remove that fraction (e.g. `0.5` strips half of it).

Why not always go to `1.0`? Because some of a model's feature exposure is genuinely *useful* signal, not just risk — so maxing out neutralization can throw away edge along with the risk. The right amount is a trade-off, and the sweet spot is often **below 1**. Below we sweep several proportions against one group and tabulate the full metric set; watch the key numbers move: max exposure (risk) should fall toward 0, mean CORR (accuracy) should only dip a little, and ideally contribution (the AIMC proxy) holds up or rises.

In [ ]:
# --- two small helpers for neutralizing predictions against a set of features ---

def neutralizer_matrix(df, feats):
    # Build the regressors for neutralization from the chosen feature columns.
    # features_matrix turns the -1 missing sentinel into NaN; we then fill NaNs with the
    # column mean (0 as a last resort) because the least-squares solve can't accept NaNs.
    N = features_matrix(df, feats)
    return N.fillna(N.mean()).fillna(0.0)

def neutralize_per_exped(df, pred_col, feats, proportion):
    # Neutralization is a CROSS-SECTIONAL operation: it only makes sense within a single
    # exped (one day's universe). So we neutralize each exped's slice on its own, then
    # concatenate the residuals back into one Series aligned to the original rows.
    parts = [neutralize(g[pred_col], neutralizer_matrix(g, feats), proportion)
             for _, g in df.groupby(EXPED)]
    return pd.concat(parts)

# --- sweep the proportion and tabulate the full metric set per proportion ---
#   metrics() columns (mean CORR / std / Sharpe / drawdown) + contribution (AIMC proxy)
#   + max_exposure (the remaining feature risk).
sweep = {}
for p in [0.0, 0.25, 0.5, 0.75, 1.0]:
    col = f"neut_{p}"
    # proportion 0 = leave the predictions untouched; otherwise neutralize at proportion p.
    val_eval[col] = (val_eval["pred_base"] if p == 0
                     else neutralize_per_exped(val_eval, "pred_base", group_feats, p))
    sweep[p] = {**metrics(per_exped_corr(val_eval, col)),
                "contribution": contribution(val_eval, col).mean(),
                "max_exposure": val_eval[group_feats].corrwith(val_eval[col]).abs().max()}
sweep = pd.DataFrame(sweep).T
sweep.index.name = "proportion"
display(sweep)

### 4. CORR by proportion

This pictures the **accuracy** side of the table above: the cumulative validation CORR for each proportion, drawn as a rough backtest of how each neutralization strength would have tracked the target over the period. (The risk side — exposure — is the next plot.) Watch whether stronger neutralization noticeably lowers the curve or barely moves it.

In [ ]:
# Cumulative CORR over validation, one line per proportion (uses the neut_* columns above).
cum = pd.DataFrame({f"p={p}": per_exped_corr(val_eval, f"neut_{p}").cumsum()
                    for p in [0.0, 0.25, 0.5, 0.75, 1.0]})
cum.plot(title=f"Cumulative validation CORR by neutralization proportion ('{GROUP}')",
         figsize=(8, 4), xticks=[])
plt.tight_layout(); plt.show()

### 5. Exposure over time

A single max-exposure number hides whether the risk is steady or spiky. Plotting the **worst exposure each exped** for the base model vs the fully-neutralized model shows neutralization flattening that risk across the whole period — not just on average.

In [ ]:
def max_exposure_per_exped(df, pred_col, feats):
    # The single largest |feature exposure| within each exped, ordered in time.
    s = df.groupby(EXPED).apply(lambda g: g[feats].corrwith(g[pred_col]).abs().max())
    return s.sort_index(key=lambda idx: idx.map(exped_num))

pd.DataFrame({
    "base": max_exposure_per_exped(val_eval, "pred_base", group_feats),
    "neutralized (p=1)": max_exposure_per_exped(val_eval, "neut_1.0", group_feats),
}).plot(title=f"Max exposure to '{GROUP}' per exped: base vs neutralized",
        figsize=(8, 3), xticks=[], xlabel="exped", ylabel="max |exposure|")
plt.tight_layout(); plt.show()

### 6. Group comparison

Different groups carry different risk. Here we fully neutralize (`proportion=1`) against each group in turn and compare the effect on CORR and contribution — some groups help, some hurt. *(This is the slow cell: it loops over every group x every exped.)*

In [ ]:
# Try fully neutralizing (proportion=1) against EACH group in turn, recording what it does
# to accuracy (mean CORR) and to the AIMC proxy (contribution).
rows = []
for g_name in [g for g in feature_sets if g not in ("small", "medium", "all")]:
    feats = [f for f in feature_sets[g_name] if f in val_eval.columns]
    # Neutralize the base predictions against this group, into a scratch column reused each loop.
    val_eval["tmp_neut"] = neutralize_per_exped(val_eval, "pred_base", feats, 1.0)
    c = per_exped_corr(val_eval, "tmp_neut")
    rows.append({"group": g_name,
                 "mean_CORR": c.mean(),
                 "sharpe": c.mean() / c.std(),
                 "contribution": contribution(val_eval, "tmp_neut").mean()})
val_eval.drop(columns=["tmp_neut"], inplace=True)   # tidy up the scratch column

# Sort by contribution: groups ABOVE the base model's contribution are the ones whose
# removal adds differentiated signal -> the groups actually worth neutralizing.
pd.DataFrame(rows).set_index("group").sort_values("contribution", ascending=False)

**Reading it:** as `proportion` -> 1 the max exposure heads toward ~0 while CORR often only dips slightly — which is why the best proportion is rarely exactly 1. Across groups, the ones whose contribution beats the base are the ones worth neutralizing. The real payoff shows up in **AIMC and consistency**; confirm it after submitting via `run_validation_diagnostics`.

## Part B — Target ensembling

Training only on `target_everest_20` can leave signal on the table: the other targets are different (but related) definitions of forward return, and a model trained on a *diverse* one makes different mistakes — so ensembling such models can raise consistency and AIMC. (Always **rank-normalize before averaging**.) First, let's get oriented on the targets.

### 1. The targets

EIQ has **16 targets**, all forward returns but defined differently:

- **8 "peaks"** — `everest`, `k2`, `lhotse`, `makalu`, `manaslu`, `nanga_parbat`, `nuptse`, `gyachung_kang` — each a *different definition* of forward return.
- **2 horizons** each — the `_20` / `_60` suffix is how many trading days ahead the return is measured.

**`target_everest_20` is the main, scored target** — the only one the payout is computed on. The other 15 are **auxiliary**: never scored, but useful differently-flavored signals to train on and ensemble. (The data also carries a generic `target` column that is *exactly equal* to `target_everest_20` — a convenience alias.)

In [ ]:
# Lay the targets out as peak x horizon; the scored one is target_everest_20.
peaks = sorted({t.replace("target_", "").rsplit("_", 1)[0] for t in target_cols})
layout = pd.DataFrame({
    "_20 horizon": {p: f"target_{p}_20" for p in peaks},
    "_60 horizon": {p: f"target_{p}_60" for p in peaks},
})
print("MAIN / scored target:", PRIMARY_TARGET)
if "target" in train.columns:                  # confirm the alias claim at runtime
    print("generic 'target' == target_everest_20:", bool(train["target"].equals(train[PRIMARY_TARGET])))
layout

### 2. Target correlation matrix

Before picking any to ensemble, it helps to see how the targets relate. The **correlation matrix** below shows every pair: brighter cells are near-duplicate targets (carrying much the same information), darker (lower) cells are pairs that behave more differently. Targets that move *differently* are the interesting ones for an ensemble — their errors partly cancel — though, as we'll see, being different isn't enough on its own: a target also has to predict something.

In [ ]:
# Correlation between every pair of targets. Bright cells = near-duplicate targets;
# darker (lower) cells = targets that behave more differently.
corr_mat = train[target_cols].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr_mat.values, cmap="coolwarm", vmin=0, vmax=1)
ax.set_xticks(range(len(target_cols))); ax.set_xticklabels(target_cols, rotation=90, fontsize=6)
ax.set_yticks(range(len(target_cols))); ax.set_yticklabels(target_cols, fontsize=6)
fig.colorbar(im, fraction=0.046, pad=0.04); ax.set_title("Target correlation matrix")
plt.tight_layout(); plt.show()

### 3. Correlation with the scored target

The full matrix shows every pair; a particularly useful view is each target's correlation with **`target_everest_20`** itself. Targets near the top are near-duplicates of the scored target (little new information); those lower down — even negatively correlated ones — are the diverse candidates that *might* add something, provided they also predict well.

In [ ]:
# Each target's correlation with the scored target: top = redundant, bottom = most diverse.
(train[target_cols].corrwith(train[PRIMARY_TARGET])
    .sort_values(ascending=False)
    .to_frame("corr_with_everest_20"))

### 4. Picking auxiliaries

Choosing *which* auxiliaries to ensemble well is its own topic and beyond this tutorial. To keep things concrete, we'll just use three of the 20-day targets — **`target_k2_20`**, **`target_makalu_20`**, and **`target_manaslu_20`** — alongside the scored target. They're a deliberate mix: `k2` and `makalu` are strongly correlated with `everest`, while `manaslu` behaves quite differently — so the evaluation below should show some adding more than others. Swap in any other targets to experiment.

In [ ]:
# For this tutorial we simply use three 20-day auxiliary targets alongside the scored one.
# (Swap in any others from the schema to experiment.)
candidates = [PRIMARY_TARGET, "target_k2_20", "target_makalu_20", "target_manaslu_20"]
print("Candidates:", candidates)

### 5. Evaluate the candidates

Our choice so far is just illustrative — it says nothing about whether these targets are actually *useful* — so we train a model on each and measure how it actually performs: **mean CORR**, **Sharpe** (consistency), **max drawdown** (worst peak-to-trough), and **contribution** (the AIMC proxy), alongside its correlation with the scored target.

In [ ]:
# Train one model per candidate target, predict on validation.
X_train = features_matrix(train, feature_set)
X_val = features_matrix(val_eval, feature_set)
for t in candidates:
    val_eval[f"pred_{t}"] = make_model().fit(X_train, train[t]).predict(X_val)

# Performance of each candidate (metrics() is defined in setup, so the columns match elsewhere).
candidate_table = pd.DataFrame({
    t: {**metrics(per_exped_corr(val_eval, f"pred_{t}")),
        "contribution": contribution(val_eval, f"pred_{t}").mean(),
        "corr_with_primary": train[t].corr(train[PRIMARY_TARGET])}
    for t in candidates
}).T
candidate_table

In [ ]:
# A side-by-side backtest of the single-target models: cumulative CORR per candidate.
pd.DataFrame({t: per_exped_corr(val_eval, f"pred_{t}").cumsum() for t in candidates}).plot(
    title="Cumulative validation CORR by candidate target", figsize=(8, 4), xticks=[])
plt.tight_layout(); plt.show()

### 6. Pairwise ensembles

Following Numerai's approach, we build **one ensemble per auxiliary** — each a two-model combination of the **scored-target model** and a single **auxiliary's model**. The `weighted_ensemble` helper below rank-normalizes each member within every exped (putting them on the same 0-1 scale), then takes a **weighted average**.

The weights live in a plain dict, so they're easy to change. **We use 50/50 for this demonstration**, but you can set any positive numbers (they're normalized for you) — e.g. to favour the scored target, use `{"pred_base": 0.7, f"pred_{aux}": 0.3}`. That makes this a reusable building block: compute whatever weights you like and drop them in.

In [ ]:
# Reusable helper: combine member models by a weighted average of their per-exped ranks.
# `weights` maps a prediction column -> its weight. Weights are normalized, so any positive
# numbers work — change them to reweight the ensemble.
def weighted_ensemble(df, weights):
    cols = list(weights)
    w = np.array([weights[c] for c in cols], dtype=float)
    w = w / w.sum()                                  # normalize the weights to sum to 1
    ranks = df.groupby(EXPED)[cols].rank(pct=True)   # put each model on the same 0-1 scale
    return ranks.to_numpy() @ w                      # weighted average -> one prediction column

# One ensemble per auxiliary. The weights dict is the knob to change (50/50 by default).
auxiliaries = [t for t in candidates if t != PRIMARY_TARGET]
ensembles = {}                                       # display-name -> column name
for aux in auxiliaries:
    weights = {"pred_base": 0.5, f"pred_{aux}": 0.5}  # <-- edit these to reweight the pairing
    val_eval[f"ens_{aux}"] = weighted_ensemble(val_eval, weights)
    ensembles[f"everest_20 + {aux.replace('target_', '')}"] = f"ens_{aux}"

print("Built pairwise ensembles:", list(ensembles))

### 7. Evaluate the ensembles

Compare each pairwise ensemble against the base (scored-target-only) model. Alongside CORR / Sharpe / drawdown, the summary adds a few Numerai-style columns: **std** (volatility of per-exped CORR), **corr_with_base** (how far the ensemble drifted from the scored-target model — lower = more differentiated), and an **estimated payout** from EIQ's formula `0.75 * CORR + 2.25 * contribution`.

> The estimated payout is an **approximation** — `contribution` is our offline BMC-style proxy, not the true AIMC (computed server-side). Use it to *rank* variants, not as a literal payout figure.

In [ ]:
# Per-variant metrics, including an estimated payout from EIQ's payout formula.
variants = {"base (everest_20)": "pred_base", **ensembles}
rows = {}
for name, col in variants.items():
    c = per_exped_corr(val_eval, col)
    contr = contribution(val_eval, col).mean()
    rows[name] = {**metrics(c),
                  "contribution": contr,
                  "corr_with_base": per_exped_corr(val_eval, col, "pred_base").mean(),
                  "est_payout": 0.75 * c.mean() + 2.25 * contr}   # 0.75*CORR + 2.25*AIMC (proxy)
ensemble_summary = pd.DataFrame(rows).T
display(ensemble_summary)

# Two backtests side by side: cumulative CORR (accuracy) and cumulative contribution (AIMC proxy).
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
pd.DataFrame({n: per_exped_corr(val_eval, c).cumsum() for n, c in variants.items()}).plot(
    ax=ax1, title="Cumulative CORR", xticks=[])
pd.DataFrame({n: contribution(val_eval, c).cumsum() for n, c in variants.items()}).plot(
    ax=ax2, title="Cumulative contribution (AIMC proxy)", xticks=[])
plt.tight_layout(); plt.show()

### 8. Benchmark comparison

EIQ publishes the benchmark models that the consensus is built from. A quick sanity check: is our best ensemble beating the best *single* benchmark on CORR, over the same expeds? (They're strong models — don't expect to win immediately; that's the bar.)

In [ ]:
# Align the benchmark predictions to our evaluation rows, then score each benchmark on CORR.
bench_eval = bench.merge(val_eval[["id", EXPED, PRIMARY_TARGET]], on=["id", EXPED], how="inner")
bench_corr = {c: per_exped_corr(bench_eval, c).mean() for c in bench_cols}
best_bench = max(bench_corr, key=bench_corr.get)
our_best = ensemble_summary["mean_CORR"].drop("base (everest_20)").max()
print(f"Best benchmark model : {best_bench}  (mean CORR {bench_corr[best_bench]:.4f})")
print(f"Our best ensemble    : mean CORR {our_best:.4f}")
print("-> beating the best benchmark" if our_best > bench_corr[best_bench]
      else "-> not beating the best benchmark yet")

**Reading it:** the auxiliaries that genuinely *differ* from `everest_20` (here, `manaslu`) are the ones most likely to lift contribution and the estimated payout when paired in; near-duplicates (`k2`, `makalu` — both ~0.86-0.90 correlated with the scored target) tend to change little. The **est_payout** column is the bottom line to optimize (CORR weighted 0.75, contribution 2.25 — watch it more than raw CORR). Knobs to experiment with: **which** auxiliaries you pair in and the **weights** (we used 50/50). Beating the benchmarks is the real bar, and it takes work.

## Submitting a neutralized / ensembled model

Both techniques just produce a prediction column (`neut_1.0`, `pred_equal`, `pred_weighted`, or a neutralized ensemble — you can stack them). To submit, do what `hello_eiq` Section 8 does: build `{instrument_id: prediction}` from your chosen column over the **live** set, `create_model`, then `submit_futures_predictions`. For automated daily runs, wrap the pipeline and `upload_model` a `.pkl`.

**Then** confirm the real AIMC across your variants with `client.run_validation_diagnostics(model_id=...)` once they've scored — the offline contribution proxy points the direction, but the live consensus is the final word.